# Week 9 — Module 20: Bayesian optimisation and diminishing returns

This week evaluates optimisation progress across eight functions, using Gaussian Process (GP) regression, neural-network (NN) ensembles, acquisition functions and local refinement. Recent improvements are examined to assess whether returns are diminishing.

## 1. Load observations

Load the cumulative observations for the eight functions.

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

search_roots = [Path.cwd(), *Path.cwd().parents]
relative_locations = [
    Path("8. Module 19/updated_data_week_8"),
    Path("updated_data_week_8"),
    Path("9. Module 20/Data"),
    Path("8. Module 19/Data"),
    Path("Data"),
]
data_folder = next(
    (base / location for base in search_roots for location in relative_locations
     if all((base / location / f"function_{i}.csv").is_file() for i in range(1, 9))),
    None,
)
if data_folder is None:
    raise FileNotFoundError(
        "Could not find all eight Week 8 function CSVs. "
        "Use updated_data_week_8 or the Data folder in Module 19/20, "
        "as with the Week 8 notebook."
    )

data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}
print(f"Loaded eight datasets from {data_folder}")
display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]), "Dimensions": data[i].shape[1] - 1}
    for i in range(1, 9)
]).set_index("Function"))

Loaded eight datasets from C:\Users\charl\OneDrive - thesuperyachtgroup.com\Desktop\Imperial AI course\7. Capstone Project\9. Module 20\Data


,Observations,Dimensions
Function,,
F1,18,2
F2,18,2
F3,23,3
F4,38,4
F5,28,4
F6,28,5
F7,38,6
F8,48,8


## 2. Week 8 observations

Compare the Week 8 outputs with the previous best for each function and update the observations.

In [3]:
# Week 8 submitted points
week_8_points = {
    1: [0.998294, 0.000709],
    2: [0.697103, 0.994806],
    3: [0.463879, 0.497888, 0.452731],
    4: [0.386248, 0.381010, 0.343907, 0.423092],
    5: [1.000000, 1.000000, 0.970000, 1.000000],
    6: [0.426973, 0.384648, 0.468006, 0.752822, 0.084308],
    7: [0.157801, 0.305871, 0.468629, 0.230835, 0.339422, 0.619025],
    8: [0.026226, 0.080912, 0.128250, 0.056836,
        0.748250, 0.399531, 0.253527, 0.720750]
}

# Outputs returned for the Week 8 queries
week_8_outputs = {
    1: 0.0,
    2: 0.6571201027872348,
    3: -0.0008627356442611198,
    4: 0.3872619363215759,
    5: 8115.434723142181,
    6: -0.2743879717132347,
    7: 2.922059249051193,
    8: 9.957031536789
}

In [4]:
assessment = []
for i in range(1, 9):
    df = data[i]
    X = df.iloc[:, :-1].to_numpy(dtype=float)
    y = df.iloc[:, -1].to_numpy(dtype=float)
    point = np.asarray(week_8_points[i])
    matches = np.all(np.isclose(X, point, atol=5e-07, rtol=0), axis=1)
    if matches.any():
        historical = y[~matches]
        if not np.isclose(y[matches][-1], week_8_outputs[i], atol=1e-07, rtol=1e-08):
            raise ValueError(f'F{i}: existing Week 8 point has inconsistent output')
    else:
        historical = y
    previous_best = historical.max()
    new_output = week_8_outputs[i]
    assessment.append({'Function': f'F{i}', 'Previous best': previous_best, 'Week 8 output': new_output, 'Difference': new_output - previous_best, 'New best?': bool(new_output > previous_best)})
assessment_table = pd.DataFrame(assessment).set_index("Function")
display(assessment_table.style.format({"Previous best":"{:.6f}","Week 8 output":"{:.6f}","Difference":"{:+.6f}"}))

,Previous best,Week 8 output,Difference,New best?
Function,,,,
F1,0.000000,0.000000,-0.000000,False
F2,0.746962,0.657120,-0.089842,False
F3,-0.010190,-0.000863,+0.009327,True
F4,0.186912,0.387262,+0.200350,True
F5,8662.482500,8115.434723,-547.047777,False
F6,-0.379740,-0.274388,+0.105352,True
F7,2.929370,2.922059,-0.007311,False
F8,9.930811,9.957032,+0.026220,True


In [5]:
for i in range(1, 9):
    x_cols, y_col = (data[i].columns[:-1], data[i].columns[-1])
    X = data[i].loc[:, x_cols].to_numpy(dtype=float)
    point = np.asarray(week_8_points[i])
    present = np.any(np.all(np.isclose(X, point, atol=5e-07, rtol=0), axis=1))
    if not present:
        row = dict(zip(x_cols, week_8_points[i]))
        row[y_col] = week_8_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([row])], ignore_index=True)
display(pd.DataFrame([{"Function":f"F{i}","Observations":len(data[i]),"Dimensions":data[i].shape[1]-1} for i in range(1,9)]).set_index("Function"))
expected_counts = {1: 18, 2: 18, 3: 23, 4: 38, 5: 28, 6: 28, 7: 38, 8: 48}
for i in range(1, 9):
    if len(data[i]) != expected_counts[i]:
        raise ValueError(
            f"F{i}: expected {expected_counts[i]} observations after Week 8 update, "
            f"found {len(data[i])} in {data_folder}. Check the source CSVs."
        )


,Observations,Dimensions
Function,,
F1,18,2
F2,18,2
F3,23,3
F4,38,4
F5,28,4
F6,28,5
F7,38,6
F8,48,8


## 3. Running-best progress and diminishing returns

Running-best values and recent improvement sizes are examined to assess how progress varies between functions. Very small F1 results are retained in scientific notation.

In [6]:
for i in range(1, 9):
    y_col = data[i].columns[-1]
    y = data[i][y_col].to_numpy()
    running_best = np.maximum.accumulate(y)
    improvement_idx = np.where(np.r_[True, running_best[1:] > running_best[:-1]])[0]

scaling_rows = []
for i in range(1, 9):
    y = data[i]["output"].to_numpy(dtype=float)
    running_best = np.maximum.accumulate(y)
    improvement_idx = np.where(np.r_[True, running_best[1:] > running_best[:-1]])[0]
    scaling_rows.append({"Function":f"F{i}","Observations":len(y),"Current best":running_best[-1],"Recorded bests":len(improvement_idx),"Last best at obs":int(improvement_idx[-1]+1)})
scaling_table = pd.DataFrame(scaling_rows).set_index("Function")
display(scaling_table.style.format({"Current best":"{:.6g}"}))

,Observations,Current best,Recorded bests,Last best at obs
Function,,,,
F1,18,7.71088e-16,3,3
F2,18,0.746962,3,15
F3,23,-0.000862736,8,23
F4,38,0.387262,11,38
F5,28,8662.48,11,27
F6,28,-0.274388,5,28
F7,38,2.92937,6,36
F8,48,9.95703,9,48


In [7]:
for i in range(1, 9):
    y_col = data[i].columns[-1]
    y = data[i][y_col].to_numpy()
    running_best = np.maximum.accumulate(y)
    improvements = []
    for j in range(1, len(running_best)):
        if running_best[j] > running_best[j - 1]:
            previous_best = running_best[j - 1]
            new_best = running_best[j]
            gain = new_best - previous_best
            improvements.append((j + 1, previous_best, new_best, gain))
    if len(improvements) == 0:
        continue

improvement_rows = []
for i in range(1, 9):
    rb = np.maximum.accumulate(data[i]["output"].to_numpy(dtype=float))
    indices = np.where(rb[1:] > rb[:-1])[0] + 1
    for idx in indices[-5:]:
        improvement_rows.append({"Function":f"F{i}","Observation":int(idx+1),"Previous best":rb[idx-1],"New best":rb[idx],"Gain":rb[idx]-rb[idx-1]})
improvement_table = pd.DataFrame(improvement_rows).set_index(["Function","Observation"])
display(improvement_table.style.format({"Previous best":"{:.6g}","New best":"{:.6g}","Gain":"{:.6g}"}))

## 4. Gaussian Process models and diagnostics

RBF Gaussian Processes are fitted to standardised outputs using 10 optimiser restarts. The fitted length scales and log marginal likelihoods are used to assess model behaviour. F1 and F2 remain at the lower length-scale bound, limiting confidence in their predictions.

In [8]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.preprocessing import StandardScaler
import numpy as np
gp_models = {}
gp_scalers = {}
for i in range(1, 9):
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()
    kernel = RBF(length_scale=0.2, length_scale_bounds=(1e-05, 100000.0))
    gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-06, normalize_y=False, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y_scaled)
    gp_models[i] = gp
    gp_scalers[i] = scaler

gp_diagnostics = pd.DataFrame([{"Function":f"F{i}","Observations":len(data[i]),"RBF length scale":float(gp_models[i].kernel_.length_scale),"Log marginal likelihood":float(gp_models[i].log_marginal_likelihood(gp_models[i].kernel_.theta)),"At lower bound?":float(gp_models[i].kernel_.length_scale)<=1.000001e-5} for i in range(1,9)]).set_index("Function")
display(gp_diagnostics.style.format({"RBF length scale":"{:.6g}","Log marginal likelihood":"{:.6f}"}))

,Observations,RBF length scale,Log marginal likelihood,At lower bound?
Function,,,,
F1,18,1e-05,-25.540894,True
F2,18,1e-05,-25.540894,True
F3,23,0.201124,-25.852938,False
F4,38,0.313209,-23.818726,False
F5,28,0.303135,-23.970400,False
F6,28,0.527186,-22.245983,False
F7,38,0.323401,-34.151061,False
F8,48,1.05365,-21.138922,False


## 5. Neural-network leave-one-out validation

Leave-one-out cross-validation (LOOCV) compares an individual neural network's mean absolute error with a mean-prediction baseline. Each training fold is scaled independently. Two hidden layers of 50 neurons are used, compared with 20 in Weeks 7–8; cross-week validation differences therefore also reflect this architecture change.

In [9]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut
import numpy as np
nn_results = {}
for i in range(1, 9):
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values
    loo = LeaveOneOut()
    nn_predictions = []
    baseline_predictions = []
    actual_values = []
    for train_index, test_index in loo.split(X):
        X_train, X_test = (X[train_index], X[test_index])
        y_train, y_test = (y[train_index], y[test_index])
        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        nn = MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=42)
        nn.fit(X_train_scaled, y_train_scaled)
        pred_scaled = nn.predict(X_test_scaled)
        pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()[0]
        baseline_pred = np.mean(y_train)
        nn_predictions.append(pred)
        baseline_predictions.append(baseline_pred)
        actual_values.append(y_test[0])
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

nn_validation_table = pd.DataFrame([{"Function":f"F{i}","Observations":len(data[i]),"NN LOOCV MAE":nn_results[i]["nn_mae"],"Baseline MAE":nn_results[i]["baseline_mae"],"Beats baseline?":nn_results[i]["beats_baseline"]} for i in range(1,9)]).set_index("Function")
display(nn_validation_table.style.format({"NN LOOCV MAE":"{:.6f}","Baseline MAE":"{:.6f}"}))

,Observations,NN LOOCV MAE,Baseline MAE,Beats baseline?
Function,,,,
F1,18,0.001837,0.000995,False
F2,18,0.236724,0.241635,True
F3,23,0.056881,0.053409,False
F4,38,1.385355,7.460801,True
F5,28,255.307417,2324.264043,True
F6,28,0.255144,0.530835,True
F7,38,0.443789,0.712291,True
F8,48,0.309128,0.981238,True


## 6. GP acquisition candidates

For each function, 100,000 candidate points are assessed using UCB (κ = 0.5, 1.0, 1.5 and 2.5) and EI (ξ = 0%, 1%, 5% and 10% of the output standard deviation). Predicted outputs, uncertainties and distances to existing observations are compared.

In [10]:
from scipy.stats import norm
from scipy.spatial.distance import cdist
n_candidates = 100000
rng = np.random.default_rng(42)
kappa_values = [0.5, 1.0, 1.5, 2.5]
xi_fractions = [0.0, 0.01, 0.05, 0.1]
gp_candidate_results = {}
for i in range(1, 9):
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values
    n_dimensions = X.shape[1]
    X_candidates = rng.random((n_candidates, n_dimensions))
    mu_scaled, sigma_scaled = gp_models[i].predict(X_candidates, return_std=True)
    scaler = gp_scalers[i]
    mu = scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
    sigma = sigma_scaled * scaler.scale_[0]
    nearest_distances = cdist(X_candidates, X).min(axis=1)
    current_best = np.max(y)
    output_sd = np.std(y)
    function_results = {}
    for kappa in kappa_values:
        ucb = mu + kappa * sigma
        idx = np.argmax(ucb)
        function_results[f'UCB_{kappa}'] = {'candidate': X_candidates[idx], 'predicted': mu[idx], 'uncertainty': sigma[idx], 'acquisition': ucb[idx], 'nearest_distance': nearest_distances[idx]}
    for xi_fraction in xi_fractions:
        xi = xi_fraction * output_sd
        improvement = mu - current_best - xi
        with np.errstate(divide='ignore', invalid='ignore'):
            Z = np.divide(improvement, sigma, out=np.zeros_like(improvement), where=sigma > 0)
            ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0] = 0
        idx = np.argmax(ei)
        label = f'EI_{int(xi_fraction * 100)}pct'
        function_results[label] = {'candidate': X_candidates[idx], 'predicted': mu[idx], 'uncertainty': sigma[idx], 'acquisition': ei[idx], 'nearest_distance': nearest_distances[idx], 'xi': xi}
    gp_candidate_results[i] = function_results

def coordinates(point):
    return "[" + ", ".join(f"{value:.6f}" for value in np.asarray(point)) + "]"

acquisition_rows = []
for i in range(1, 9):
    for method, result in gp_candidate_results[i].items():
        acquisition_rows.append({"Function":f"F{i}","Method":method,"Candidate":coordinates(result["candidate"]),"GP prediction":result["predicted"],"GP uncertainty":result["uncertainty"],"Acquisition score":result["acquisition"],"Nearest distance":result["nearest_distance"]})
acquisition_table = pd.DataFrame(acquisition_rows)
for prefix, label in [("UCB_","UCB comparison"),("EI_","Expected Improvement comparison")]:
    print(label)
    display(acquisition_table[acquisition_table["Method"].str.startswith(prefix)].set_index(["Function","Method"]).style.format({"GP prediction":"{:.5g}","GP uncertainty":"{:.5g}","Acquisition score":"{:.5g}","Nearest distance":"{:.4f}"}))

UCB comparison


Expected Improvement comparison


## 7. Neural-network ensemble candidates

Five NN models with `(50, 50)` hidden layers and random seeds 42–46 are fitted to each function. Ensemble predictions and disagreement are compared across 100,000 candidate points.

In [11]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import numpy as np
n_candidates = 100000
seeds = [42, 43, 44, 45, 46]
nn_candidate_results = {}
nn_models = {}
for i in range(1, 9):
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values
    n_dimensions = X.shape[1]
    x_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()
    models = []
    for seed in seeds:
        nn = MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=seed)
        nn.fit(X_scaled, y_scaled)
        models.append(nn)
    nn_models[i] = {'models': models, 'x_scaler': x_scaler, 'y_scaler': y_scaler}
    rng = np.random.default_rng(42 + i)
    X_candidates = rng.random((n_candidates, n_dimensions))
    X_candidates_scaled = x_scaler.transform(X_candidates)
    predictions = []
    for model in models:
        pred_scaled = model.predict(X_candidates_scaled)
        pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        predictions.append(pred)
    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    disagreement = predictions.std(axis=0)
    idx = np.argmax(mean_prediction)
    candidate = X_candidates[idx]
    predicted_output = mean_prediction[idx]
    candidate_disagreement = disagreement[idx]
    nearest_distance = cdist(candidate.reshape(1, -1), X).min()
    nn_candidate_results[i] = {'candidate': candidate, 'predicted': predicted_output, 'disagreement': candidate_disagreement, 'nearest_distance': nearest_distance}

nn_candidate_table = pd.DataFrame([{"Function":f"F{i}","Candidate":coordinates(nn_candidate_results[i]["candidate"]),"Predicted output":nn_candidate_results[i]["predicted"],"Ensemble disagreement":nn_candidate_results[i]["disagreement"],"Nearest distance":nn_candidate_results[i]["nearest_distance"],"LOOCV validated?":nn_results[i]["beats_baseline"]} for i in range(1,9)]).set_index("Function")
display(nn_candidate_table.style.format({"Predicted output":"{:.6f}","Ensemble disagreement":"{:.6f}","Nearest distance":"{:.6f}"}))

,Candidate,Predicted output,Ensemble disagreement,Nearest distance,LOOCV validated?
Function,,,,,
F1,"[0.517917, 0.000290]",0.001946,0.001331,0.182315,False
F2,"[0.702517, 0.009487]",0.831771,0.431009,0.106007,True
F3,"[0.465542, 0.513952, 0.462107]",-0.001013,0.000717,0.018674,False
F4,"[0.386457, 0.364301, 0.315578, 0.423391]",0.173431,0.101814,0.032892,True
F5,"[0.996673, 0.749297, 0.983441, 0.989318]",8132.577652,837.663492,0.247618,True
F6,"[0.476200, 0.401879, 0.450627, 0.837966, 0.006748]",-0.248980,0.042060,0.127622,True
F7,"[0.150161, 0.338209, 0.511025, 0.190914, 0.382394, 0.542431]",2.893229,0.055542,0.110492,True
F8,"[0.012982, 0.180599, 0.189901, 0.256228, 0.760257, 0.471612, 0.155487, 0.557332]",10.056825,0.027228,0.234486,True


## 8. GP–NN agreement

Euclidean distances between the NN recommendation and GP acquisition candidates are used to assess agreement between the surrogate models.

In [12]:
gp_nn_agreement = {}
for i in range(1, 9):
    nn_candidate = nn_candidate_results[i]['candidate']
    comparisons = []
    for method, result in gp_candidate_results[i].items():
        gp_candidate = result['candidate']
        distance = np.linalg.norm(nn_candidate - gp_candidate)
        comparisons.append((method, distance, gp_candidate))
    comparisons.sort(key=lambda x: x[1])
    closest_method, closest_distance, closest_candidate = comparisons[0]
    gp_nn_agreement[i] = {'closest_method': closest_method, 'distance': closest_distance, 'gp_candidate': closest_candidate, 'nn_candidate': nn_candidate}

agreement_table = pd.DataFrame([{"Function":f"F{i}","Closest GP method":gp_nn_agreement[i]["closest_method"],"GP–NN distance":gp_nn_agreement[i]["distance"],"NN candidate":coordinates(gp_nn_agreement[i]["nn_candidate"]),"GP candidate":coordinates(gp_nn_agreement[i]["gp_candidate"])} for i in range(1,9)]).set_index("Function")
display(agreement_table.style.format({"GP–NN distance":"{:.6f}"}))

,Closest GP method,GP–NN distance,NN candidate,GP candidate
Function,,,,
F1,UCB_0.5,1.063404,"[0.517917, 0.000290]","[0.094177, 0.975622]"
F2,UCB_1.5,0.485232,"[0.702517, 0.009487]","[0.403694, 0.391789]"
F3,UCB_0.5,0.058180,"[0.465542, 0.513952, 0.462107]","[0.520345, 0.502063, 0.477604]"
F4,UCB_0.5,0.129899,"[0.386457, 0.364301, 0.315578, 0.423391]","[0.368932, 0.317666, 0.402753, 0.340976]"
F5,UCB_1.5,0.221396,"[0.996673, 0.749297, 0.983441, 0.989318]","[0.967199, 0.967647, 0.962384, 0.994545]"
F6,EI_0pct,0.193602,"[0.476200, 0.401879, 0.450627, 0.837966, 0.006748]","[0.530285, 0.324116, 0.577827, 0.727035, 0.001859]"
F7,UCB_0.5,0.125569,"[0.150161, 0.338209, 0.511025, 0.190914, 0.382394, 0.542431]","[0.174885, 0.324058, 0.479146, 0.269144, 0.331135, 0.614490]"
F8,UCB_0.5,0.243999,"[0.012982, 0.180599, 0.189901, 0.256228, 0.760257, 0.471612, 0.155487, 0.557332]","[0.067187, 0.137254, 0.157690, 0.059914, 0.817193, 0.378654, 0.206619, 0.531946]"


## 9. Neighbourhood evidence

The three nearest observed outputs are examined for each NN recommendation and the GP UCB κ = 0.5 and κ = 2.5 candidates. This helps distinguish well-supported local regions from uncertain exploration.

In [13]:
neighbourhood_results = {}
for i in range(1, 9):
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values
    candidates = {'NN': nn_candidate_results[i]['candidate'], 'UCB_0.5': gp_candidate_results[i]['UCB_0.5']['candidate'], 'UCB_2.5': gp_candidate_results[i]['UCB_2.5']['candidate']}
    function_results = {}
    for method, candidate in candidates.items():
        distances = np.linalg.norm(X - candidate, axis=1)
        nearest_idx = np.argsort(distances)[:3]
        function_results[method] = {'candidate': candidate, 'distances': distances[nearest_idx], 'outputs': y[nearest_idx]}
    neighbourhood_results[i] = function_results

neighbourhood_rows = []
for i in range(1, 9):
    for method, result in neighbourhood_results[i].items():
        neighbourhood_rows.append({"Function":f"F{i}","Method":method,"Candidate":coordinates(result["candidate"]),"Nearest distances":", ".join(f"{v:.4f}" for v in result["distances"]),"Neighbour outputs":", ".join(f"{v:.6g}" for v in result["outputs"])})
neighbourhood_table = pd.DataFrame(neighbourhood_rows).set_index(["Function","Method"])
display(neighbourhood_table)

Candidate  \
Function Method                                                       
F1       NN                                    [0.517917, 0.000290]   
         UCB_0.5                               [0.094177, 0.975622]   
         UCB_2.5                               [0.094177, 0.975622]   
F2       NN                                    [0.702517, 0.009487]   
         UCB_0.5                               [0.697103, 0.994806]   
         UCB_2.5                               [0.403694, 0.391789]   
F3       NN                          [0.465542, 0.513952, 0.462107]   
         UCB_0.5                     [0.520345, 0.502063, 0.477604]   
         UCB_2.5                     [0.998390, 0.992191, 0.910475]   
F4       NN                [0.386457, 0.364301, 0.315578, 0.423391]   
         UCB_0.5           [0.368932, 0.317666, 0.402753, 0.340976]   
         UCB_2.5           [0.459956, 0.058548, 0.203739, 0.003179]   
F5       NN                [0.996673, 0.749297, 0.983441, 0.989318]   
         UCB_0.5           [0.939869, 0.992261, 0.978815, 0.999723]   
         UCB_2.5           [0.019672, 0.003040, 0.043324, 0.976161]   
F6       NN       [0.476200, 0.401879, 0.450627, 0.837966, 0.006...   
         UCB_0.5  [0.393353, 0.339159, 0.672210, 0.789817, 0.010...   
         UCB_2.5  [0.389969, 0.183911, 0.869622, 0.984764, 0.000...   
F7       NN       [0.150161, 0.338209, 0.511025, 0.190914, 0.382...   
         UCB_0.5  [0.174885, 0.324058, 0.479146, 0.269144, 0.331...   
         UCB_2.5  [0.283858, 0.099702, 0.458423, 0.332012, 0.410...   
F8       NN       [0.012982, 0.180599, 0.189901, 0.256228, 0.760...   
         UCB_0.5  [0.067187, 0.137254, 0.157690, 0.059914, 0.817...   
         UCB_2.5  [0.441069, 0.324388, 0.061659, 0.803622, 0.781...   

                       Nearest distances  \
Function Method                            
F1       NN       0.1823, 0.2197, 0.4170   
         UCB_0.5  0.0953, 0.3098, 0.4896   
         UCB_2.5  0.0953, 0.3098, 0.4896   
F2       NN       0.1060, 0.1202, 0.1291   
         UCB_0.5  0.0000, 0.0685, 0.0772   
         UCB_2.5  0.1134, 0.1894, 0.2645   
F3       NN       0.0187, 0.0575, 0.0996   
         UCB_0.5  0.0618, 0.0928, 0.1508   
         UCB_2.5  0.3692, 0.6919, 0.8088   
F4       NN       0.0329, 0.0350, 0.0760   
         UCB_0.5  0.1205, 0.1252, 0.1338   
         UCB_2.5  0.4165, 0.4802, 0.5106   
F5       NN       0.2476, 0.2498, 0.2513   
         UCB_0.5  0.0000, 0.0286, 0.0613   
         UCB_2.5  0.6863, 0.7711, 0.8151   
F6       NN       0.1276, 0.1840, 0.2188   
         UCB_0.5  0.2276, 0.2491, 0.2649   
         UCB_2.5  0.4709, 0.5120, 0.5136   
F7       NN       0.1105, 0.1256, 0.2049   
         UCB_0.5  0.0000, 0.0479, 0.1727   
         UCB_2.5  0.3128, 0.3136, 0.3489   
F8       NN       0.2345, 0.2479, 0.3088   
         UCB_0.5  0.2208, 0.2405, 0.2877   
         UCB_2.5  0.7994, 0.8438, 0.8652   

                                         Neighbour outputs  
Function Method                                             
F1       NN       -2.15925e-54, -2.08909e-91, 3.34177e-124  
         UCB_0.5               0, 1.32268e-79, 1.03308e-46  
         UCB_2.5               0, 1.32268e-79, 1.03308e-46  
F2       NN                   0.471844, 0.538996, 0.746962  
         UCB_0.5                0.65712, 0.611205, 0.36601  
         UCB_2.5          0.214965, -0.0138576, -0.0656236  
F3       NN           -0.000862736, -0.0101899, -0.0164513  
         UCB_0.5      -0.000862736, -0.0101899, -0.0164513  
         UCB_2.5          -0.0567584, -0.113869, -0.105965  
F4       NN                  0.387262, 0.186912, -0.200431  
         UCB_0.5             0.387262, 0.186912, -0.200431  
         UCB_2.5              -13.5276, -10.0696, -4.02554  
F5       NN                      6691.93, 7121.15, 8115.43  
         UCB_0.5                 7121.15, 6691.93, 8115.43  
         UCB_2.5                 109.572, 233.224, 64.4434  
F6       NN                 -

## 10. Multi-radius local refinement (F3–F8)

For F3–F8, 50,000 local candidates are evaluated at radii of 0.03, 0.05 and 0.10 around the current best. Candidates are ranked using the sum of standardised GP and NN predictions, excluding near-duplicates.

In [14]:
from scipy.spatial.distance import cdist
import numpy as np
local_functions = [3, 4, 5, 6, 7, 8]
radii = [0.03, 0.05, 0.1]
n_local = 50000
local_results = {}
for i in local_functions:
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_output = y[best_idx]
    function_results = {}
    for radius in radii:
        rng = np.random.default_rng(1000 + i + int(radius * 1000))
        local_candidates = best_point + rng.uniform(-radius, radius, size=(n_local, X.shape[1]))
        local_candidates = np.clip(local_candidates, 0, 1)
        distances = cdist(local_candidates, X)
        nearest_distance = distances.min(axis=1)
        mask = nearest_distance > 0.0001
        local_candidates = local_candidates[mask]
        nearest_distance = nearest_distance[mask]
        mu_scaled, sigma_scaled = gp_models[i].predict(local_candidates, return_std=True)
        scaler = gp_scalers[i]
        gp_mu = scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
        gp_sigma = sigma_scaled * scaler.scale_[0]
        nn_info = nn_models[i]
        local_scaled = nn_info['x_scaler'].transform(local_candidates)
        nn_preds = []
        for model in nn_info['models']:
            pred_scaled = model.predict(local_scaled)
            pred = nn_info['y_scaler'].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
            nn_preds.append(pred)
        nn_preds = np.array(nn_preds)
        nn_mean = nn_preds.mean(axis=0)
        nn_disagreement = nn_preds.std(axis=0)
        gp_z = (gp_mu - np.mean(gp_mu)) / (np.std(gp_mu) + 1e-12)
        nn_z = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
        combined_score = gp_z + nn_z
        idx = np.argmax(combined_score)
        function_results[radius] = {'candidate': local_candidates[idx], 'gp_predicted': gp_mu[idx], 'gp_uncertainty': gp_sigma[idx], 'nn_predicted': nn_mean[idx], 'nn_disagreement': nn_disagreement[idx], 'nearest_distance': nearest_distance[idx], 'distance_from_best': np.linalg.norm(local_candidates[idx] - best_point)}
        result = function_results[radius]
    local_results[i] = function_results

local_rows = []
for i in local_functions:
    for radius, result in local_results[i].items():
        local_rows.append({"Function":f"F{i}","Radius":radius,"Candidate":coordinates(result["candidate"]),"GP prediction":result["gp_predicted"],"GP uncertainty":result["gp_uncertainty"],"NN prediction":result["nn_predicted"],"NN disagreement":result["nn_disagreement"],"Nearest distance":result["nearest_distance"]})
local_table = pd.DataFrame(local_rows).set_index(["Function","Radius"])
display(local_table.style.format({"GP prediction":"{:.6g}","GP uncertainty":"{:.6g}","NN prediction":"{:.6g}","NN disagreement":"{:.6g}","Nearest distance":"{:.6f}"}))

## 11. Function 5: novelty-constrained refinement

The unconstrained local search converged almost exactly on the existing best at `[1, 1, 1, 1]`. Minimum distances of 0.03, 0.05 and 0.10 are compared to balance predicted performance and novelty.

In [15]:
i = 5
X = data[i].iloc[:, :-1].values
y = data[i].iloc[:, -1].values
best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]
search_radius = 0.15
n_local = 100000
novelty_thresholds = [0.03, 0.05, 0.1]
rng = np.random.default_rng(505)
local_candidates = best_point + rng.uniform(-search_radius, search_radius, size=(n_local, X.shape[1]))
local_candidates = np.clip(local_candidates, 0, 1)
distances = cdist(local_candidates, X)
nearest_distances = distances.min(axis=1)
mu_scaled, sigma_scaled = gp_models[i].predict(local_candidates, return_std=True)
scaler = gp_scalers[i]
gp_mu = scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
gp_sigma = sigma_scaled * scaler.scale_[0]
nn_info = nn_models[i]
local_scaled = nn_info['x_scaler'].transform(local_candidates)
nn_preds = []
for model in nn_info['models']:
    pred_scaled = model.predict(local_scaled)
    pred = nn_info['y_scaler'].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    nn_preds.append(pred)
nn_preds = np.array(nn_preds)
nn_mean = nn_preds.mean(axis=0)
nn_disagreement = nn_preds.std(axis=0)
f5_novelty_results = {}
for min_distance in novelty_thresholds:
    mask = nearest_distances >= min_distance
    valid_candidates = local_candidates[mask]
    valid_gp_mu = gp_mu[mask]
    valid_gp_sigma = gp_sigma[mask]
    valid_nn_mean = nn_mean[mask]
    valid_nn_disagreement = nn_disagreement[mask]
    valid_distances = nearest_distances[mask]
    gp_z = (valid_gp_mu - valid_gp_mu.mean()) / (valid_gp_mu.std() + 1e-12)
    nn_z = (valid_nn_mean - valid_nn_mean.mean()) / (valid_nn_mean.std() + 1e-12)
    combined_score = gp_z + nn_z
    idx = np.argmax(combined_score)
    candidate = valid_candidates[idx]
    f5_novelty_results[min_distance] = {'candidate': candidate, 'gp_predicted': valid_gp_mu[idx], 'gp_uncertainty': valid_gp_sigma[idx], 'nn_predicted': valid_nn_mean[idx], 'nn_disagreement': valid_nn_disagreement[idx], 'nearest_distance': valid_distances[idx], 'distance_from_best': np.linalg.norm(candidate - best_point)}
    result = f5_novelty_results[min_distance]

f5_table = pd.DataFrame([{"Min novelty":distance,"Candidate":coordinates(result["candidate"]),"GP prediction":result["gp_predicted"],"GP uncertainty":result["gp_uncertainty"],"NN prediction":result["nn_predicted"],"NN disagreement":result["nn_disagreement"],"Nearest distance":result["nearest_distance"]} for distance,result in f5_novelty_results.items()]).set_index("Min novelty")
display(f5_table.style.format({"GP prediction":"{:.6f}","GP uncertainty":"{:.6f}","NN prediction":"{:.6f}","NN disagreement":"{:.6f}","Nearest distance":"{:.6f}"}))

,Candidate,GP prediction,GP uncertainty,NN prediction,NN disagreement,Nearest distance
Min novelty,,,,,,
0.030000,"[1.000000, 0.969998, 1.000000, 1.000000]",8287.623017,247.689521,8688.604318,118.482518,0.030002
0.050000,"[1.000000, 0.949994, 1.000000, 1.000000]",8001.755491,410.333828,8710.650195,197.196894,0.050006
0.100000,"[1.000000, 0.899981, 1.000000, 1.000000]",7187.506528,802.299569,8743.779604,369.571101,0.100019


## 12. Function 1: space-filling exploration

Both surrogate approaches remain unreliable for F1. A model-independent maximin search across 200,000 points is used to identify a less-explored region of the input space.

In [16]:
i = 1
X = data[i].iloc[:, :-1].values
rng = np.random.default_rng(101)
n_candidates = 200000
X_candidates = rng.random((n_candidates, X.shape[1]))
distances = cdist(X_candidates, X)
nearest_distances = distances.min(axis=1)
idx = np.argmax(nearest_distances)
f1_candidate = X_candidates[idx]
f1_nearest_distance = nearest_distances[idx]
nearest_idx = np.argsort(distances[idx])[:3]

f1_neighbours = pd.DataFrame([{"Neighbour rank":rank,"Observation":int(obs_idx+1),"Distance":float(distances[idx,obs_idx])} for rank,obs_idx in enumerate(nearest_idx,start=1)]).set_index("Neighbour rank")
print(f"Selected F1 point: {coordinates(f1_candidate)}")
display(f1_neighbours.style.format({"Distance":"{:.6f}"}))

Selected F1 point: [0.682178, 0.001903]


,Observation,Distance
Neighbour rank,,
1,4,0.306754
2,6,0.308314
3,18,0.316118


## 13. Function 2: NN-only refinement

F2's GP length scale remains at the lower bound. NN-only searches therefore assess 100,000 candidates at each radius of 0.03, 0.05 and 0.10 around the current best.

In [17]:
i = 2
X = data[i].iloc[:, :-1].values
y = data[i].iloc[:, -1].values
best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]
radii = [0.03, 0.05, 0.1]
n_local = 100000
f2_refinement_results = {}
for radius in radii:
    rng = np.random.default_rng(2000 + int(radius * 1000))
    candidates = best_point + rng.uniform(-radius, radius, size=(n_local, X.shape[1]))
    candidates = np.clip(candidates, 0, 1)
    nearest_distances = cdist(candidates, X).min(axis=1)
    mask = nearest_distances > 0.0001
    candidates = candidates[mask]
    nearest_distances = nearest_distances[mask]
    nn_info = nn_models[i]
    candidates_scaled = nn_info['x_scaler'].transform(candidates)
    predictions = []
    for model in nn_info['models']:
        pred_scaled = model.predict(candidates_scaled)
        pred = nn_info['y_scaler'].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        predictions.append(pred)
    predictions = np.array(predictions)
    nn_mean = predictions.mean(axis=0)
    nn_disagreement = predictions.std(axis=0)
    idx = np.argmax(nn_mean)
    candidate = candidates[idx]
    f2_refinement_results[radius] = {'candidate': candidate, 'nn_predicted': nn_mean[idx], 'nn_disagreement': nn_disagreement[idx], 'nearest_distance': nearest_distances[idx], 'distance_from_best': np.linalg.norm(candidate - best_point)}
    result = f2_refinement_results[radius]

f2_table = pd.DataFrame([{"Radius":radius,"Candidate":coordinates(result["candidate"]),"NN prediction":result["nn_predicted"],"NN disagreement":result["nn_disagreement"],"Nearest distance":result["nearest_distance"],"Distance from best":result["distance_from_best"]} for radius,result in f2_refinement_results.items()]).set_index("Radius")
display(f2_table.style.format({"NN prediction":"{:.6f}","NN disagreement":"{:.6f}","Nearest distance":"{:.6f}","Distance from best":"{:.6f}"}))

,Candidate,NN prediction,NN disagreement,Nearest distance,Distance from best
Radius,,,,,
0.030000,"[0.711375, 0.108192]",0.718309,0.149730,0.018533,0.030012
0.050000,"[0.710141, 0.088175]",0.743700,0.211086,0.030706,0.050067
0.100000,"[0.704305, 0.038527]",0.810157,0.363055,0.077676,0.100015


## 14. Final candidate selection and validation

Final points are checked for dimensionality, bounds and novelty. The selected coordinates are also compared with those recalculated by each function's search to assess reproducibility.

In [18]:
import numpy as np
from scipy.spatial.distance import cdist
final_candidates = {1: np.array([0.682178, 0.001903]), 2: np.array([0.704305, 0.038527]), 3: np.array([0.473523, 0.500295, 0.456235]), 4: np.array([0.388825, 0.366595, 0.350533, 0.420635]), 5: np.array([1.0, 0.969998, 1.0, 1.0]), 6: np.array([0.440497, 0.372759, 0.525613, 0.791828, 0.0]), 7: np.array([0.199451, 0.330731, 0.460263, 0.255214, 0.346569, 0.603374]), 8: np.array([0.031565, 0.175185, 0.160043, 0.13067, 0.742356, 0.487463, 0.194164, 0.622246])}
validation_results = {}
for i in range(1, 9):
    candidate = final_candidates[i]
    X = data[i].iloc[:, :-1].values
    expected_dimensions = X.shape[1]
    correct_dimensions = len(candidate) == expected_dimensions
    within_bounds = np.all((candidate >= 0) & (candidate <= 1))
    distances = cdist(candidate.reshape(1, -1), X).ravel()
    nearest_distance = distances.min()
    nearest_index = np.argmin(distances)
    duplicate = np.any(np.all(np.isclose(X, candidate, atol=1e-08, rtol=0), axis=1))
    passed = correct_dimensions and within_bounds and (not duplicate)
    validation_results[i] = {'dimensions': len(candidate), 'expected_dimensions': expected_dimensions, 'correct_dimensions': correct_dimensions, 'within_bounds': within_bounds, 'duplicate': duplicate, 'nearest_distance': nearest_distance, 'nearest_index': nearest_index, 'passed': passed}
all_passed = all((result['passed'] for result in validation_results.values()))
for i in range(1, 9):
    formatted = ', '.join((f'{x:.6f}' for x in final_candidates[i]))

calculated_candidates = {
    1: f1_candidate,
    2: f2_refinement_results[0.10]["candidate"],
    3: local_results[3][0.03]["candidate"],
    4: local_results[4][0.05]["candidate"],
    5: f5_novelty_results[0.03]["candidate"],
    6: local_results[6][0.10]["candidate"],
    7: local_results[7][0.03]["candidate"],
    8: local_results[8][0.10]["candidate"],
}
reproduction_rows = []
for i in range(1,9):
    reproduced = np.array_equal(np.round(calculated_candidates[i],6),np.round(final_candidates[i],6))
    reproduction_rows.append({"Function":f"F{i}","Final submission":coordinates(final_candidates[i]),"Calculated candidate":coordinates(np.round(calculated_candidates[i],6)),"Reproduced?":reproduced,"Valid?":validation_results[i]["passed"],"Nearest distance":validation_results[i]["nearest_distance"]})
reproduction_table = pd.DataFrame(reproduction_rows).set_index("Function")
display(reproduction_table.style.format({"Nearest distance":"{:.6f}"}))
print("Final submission points (six decimal places):")
for i in range(1,9):
    print(f"F{i}: " + "-".join(f"{v:.6f}" for v in final_candidates[i]))
if not reproduction_table["Valid?"].all():
    raise ValueError("One or more final points failed validation.")
if not reproduction_table["Reproduced?"].all():
    import sklearn
    print(f"Calculated candidate differs from selected point for: {', '.join(reproduction_table.index[~reproduction_table['Reproduced?']])} (scikit-learn {sklearn.__version__}).")
    print("Selected coordinates retained; compare model versions and NN diagnostics before interpreting discrepancies.")

,Final submission,Calculated candidate,Reproduced?,Valid?,Nearest distance
Function,,,,,
F1,"[0.682178, 0.001903]","[0.682178, 0.001903]",True,True,0.306755
F2,"[0.704305, 0.038527]","[0.704305, 0.038527]",True,True,0.077676
F3,"[0.473523, 0.500295, 0.456235]","[0.473523, 0.500295, 0.456235]",True,True,0.010539
F4,"[0.388825, 0.366595, 0.350533, 0.420635]","[0.388825, 0.366595, 0.350533, 0.420635]",True,True,0.016260
F5,"[1.000000, 0.969998, 1.000000, 1.000000]","[1.000000, 0.969998, 1.000000, 1.000000]",True,True,0.030002
F6,"[0.440497, 0.372759, 0.525613, 0.791828, 0.000000]","[0.440497, 0.372759, 0.525613, 0.791828, 0.000000]",True,True,0.110780
F7,"[0.199451, 0.330731, 0.460263, 0.255214, 0.346569, 0.603374]","[0.199451, 0.330731, 0.460263, 0.255214, 0.346569, 0.603374]",True,True,0.039502
F8,"[0.031565, 0.175185, 0.160043, 0.130670, 0.742356, 0.487463, 0.194164, 0.622246]","[0.031565, 0.175185, 0.160043, 0.130670, 0.742356, 0.487463, 0.194164, 0.622246]",True,True,0.190714


Final submission points (six decimal places):
F1: 0.682178-0.001903
F2: 0.704305-0.038527
F3: 0.473523-0.500295-0.456235
F4: 0.388825-0.366595-0.350533-0.420635
F5: 1.000000-0.969998-1.000000-1.000000
F6: 0.440497-0.372759-0.525613-0.791828-0.000000
F7: 0.199451-0.330731-0.460263-0.255214-0.346569-0.603374
F8: 0.031565-0.175185-0.160043-0.130670-0.742356-0.487463-0.194164-0.622246
